# Module 5: Multi-Agent (15 min)

Add agent delegation — when the customer service agent encounters a technical issue, it escalates to a **tech support specialist** agent. This is the agents-as-tools pattern: one orchestrator calls specialists like functions.

**Prerequisites:** Modules 1-4 completed

In [ ]:
!pip install -q -r requirements.txt

---

## Part 1: Build the Tech Support Specialist

A focused agent with its own tools and system prompt. It handles device troubleshooting, connectivity issues, and firmware updates.

In [ ]:
from strands import Agent, tool
from customer_service_tools import lookup_customer, get_order_history, process_refund


# --- Tech support tools ---

@tool
def check_device_compatibility(device: str, issue: str) -> str:
    """Check if a device has known compatibility issues.

    Args:
        device: The device name or model
        issue: Description of the issue
    """
    known_issues = {
        "Wireless Headphones": "Known Bluetooth 5.0 pairing issue with older devices. Fix: Reset headphones (hold power 10s), then re-pair.",
        "USB-C Hub": "Some laptops require USB-C alt mode. Check laptop specs for DisplayPort over USB-C support.",
        "Mechanical Keyboard": "Firmware v2.1 has a key ghosting bug. Update to v2.3 via manufacturer website.",
    }
    for device_name, fix in known_issues.items():
        if device_name.lower() in device.lower():
            return f"Known issue found for {device_name}: {fix}"
    return f"No known issues found for '{device}'. Recommend standard troubleshooting: restart device, check connections, update drivers."


@tool
def run_diagnostic(device: str) -> str:
    """Run a remote diagnostic check on a device.

    Args:
        device: The device name or model to diagnose
    """
    return (
        f"Diagnostic results for {device}:\n"
        f"- Firmware: v2.1 (update available: v2.3)\n"
        f"- Connection: Stable\n"
        f"- Battery: 85%\n"
        f"- Last sync: 2 hours ago\n"
        f"Recommendation: Update firmware to resolve known issues."
    )


print("✅ Tech support tools defined")

---

## Part 2: Wrap the Specialist as a Tool

The `@tool` decorator turns the specialist agent into a callable tool for the orchestrator. The orchestrator decides when to delegate.

In [ ]:
@tool
def tech_support_specialist(issue_description: str) -> str:
    """Escalate a technical issue to the tech support specialist agent.
    Use this when a customer has a device problem, connectivity issue,
    or needs technical troubleshooting beyond basic order/account help.

    Args:
        issue_description: Detailed description of the technical issue including device name and symptoms
    """
    specialist = Agent(
        tools=[check_device_compatibility, run_diagnostic],
        system_prompt="""You are a tech support specialist for an electronics store.
You diagnose device issues, check compatibility, and provide step-by-step fixes.
Be technical but clear. Always provide actionable next steps.""",
        callback_handler=None,  # Silent — don't stream to user
    )

    print(f"\n[DELEGATION] 🔧 Tech support specialist activated")
    print(f"[DELEGATION] 📋 Issue: {issue_description[:80]}...")

    response = specialist(issue_description)

    print(f"[DELEGATION] ✅ Specialist responded")
    return str(response)


print("✅ tech_support_specialist tool defined")

---

## Part 3: The Orchestrator Agent

The customer service agent now has `tech_support_specialist` as one of its tools. It decides when to escalate.

In [ ]:
SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise.

You handle:
- Account lookups and order status
- Refund processing
- Basic questions

For TECHNICAL issues (device problems, connectivity, firmware, troubleshooting),
delegate to the tech_support_specialist tool. Provide it with the device name
and a clear description of the issue.

After getting the specialist's response, relay the solution to the customer
in a friendly, non-technical way."""

orchestrator = Agent(
    tools=[lookup_customer, get_order_history, process_refund, tech_support_specialist],
    system_prompt=SYSTEM_PROMPT,
)

# This should trigger delegation to the tech support specialist
result = orchestrator(
    "I'm customer C-1001. The wireless headphones I bought aren't pairing "
    "with my phone. I've tried restarting them but nothing works."
)

In [ ]:
# This should NOT trigger delegation — it's a simple order question
orchestrator = Agent(
    tools=[lookup_customer, get_order_history, process_refund, tech_support_specialist],
    system_prompt=SYSTEM_PROMPT,
)

result = orchestrator("I'm customer C-1002. Where is my keyboard order?")

---

## 🎯 Try It Yourself

Try different scenarios to see when the orchestrator delegates vs handles directly:

In [ ]:
# Try these:
# "My USB-C hub isn't showing my external monitor" → should delegate
# "I want a refund for order ORD-5521" → should handle directly
# "The keyboard keys are sticking and some don't register" → should delegate

orchestrator = Agent(
    tools=[lookup_customer, get_order_history, process_refund, tech_support_specialist],
    system_prompt=SYSTEM_PROMPT,
)

result = orchestrator("I'm C-1002. The mechanical keyboard keys are ghosting — I press one key and two characters appear.")

---

## What's Next

The agent can now delegate to specialists. But how do you know it's working correctly at scale? In **Module 6: Evals**, you'll write automated evaluations to test the agent's behavior.